# Clinical Document Classification — Transfer Learning with BERT

Bachelor thesis: *Clinical Document Classification Using Machine Learning*

This notebook covers the **transfer learning** branch.
| Name (thesis) | Hugging Face model id |
|---|---|
| BERT Base | `google-bert/bert-base-uncased` |
| BioClinicalBERT | `emilyalsentzer/Bio_ClinicalBERT` |
| PubMedBERT | `Tsubasaz/clinical-pubmed-bert-base-128` |
| SciBERT | `allenai/scibert_scivocab_uncased` |
| ClinicalBERT | `medicalai/ClinicalBERT` |

Each checkpoint is wrapped with a linear classification head on top of mean-pooled last-hidden-state
output and fine-tuned end to end. Per the thesis, **ClinicalBERT** performs best of the five (it is
pretrained on discharge-summary notes, which share vocabulary/structure with this dataset), though
it still trails the traditional-ML Voting Classifier from notebook 1 (thesis Table 5.16: Voting
Classifier 0.89/0.888 vs. ClinicalBERT 0.85/0.84).

**Runtime note:** fine-tuning 5 transformer checkpoints is heavy — each downloads a
several-hundred-MB checkpoint and trains for multiple epochs. This is realistically a GPU (e.g.
Colab) workload; on CPU, set `MODELS_TO_RUN` below to a single entry for a smoke test rather than
running the full sweep.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics

from preprocessing import load_clinical_documents, load_spacy_model, preprocess_document

RANDOM_STATE = 133
DATA_DIR = "../data/clinical_documents"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# Thesis 5.2.5: best combination of parameters found for ClinicalBERT, reused for all variants
# for a fair comparison (given free-tier Colab's limits on batch size / epoch count).
NUM_EPOCHS = 5
BATCH_SIZE = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

BERT_MODELS = {
    "BERT Base": "google-bert/bert-base-uncased",
    "BioClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "PubMedBERT": "Tsubasaz/clinical-pubmed-bert-base-128",
    "SciBERT": "allenai/scibert_scivocab_uncased",
    "ClinicalBERT": "medicalai/ClinicalBERT",
}

# Run the full sweep, or set this to e.g. ["ClinicalBERT"] for a quick single-model run.
MODELS_TO_RUN = list(BERT_MODELS.keys())

## 1. Load & preprocess data

Same cleaning pipeline as the other two notebooks (`src/preprocessing.py`).

In [ ]:
df = load_clinical_documents(DATA_DIR)
nlp = load_spacy_model("en_core_web_lg")

df["clean_document"] = [preprocess_document(doc, nlp) for doc in df["document"]]
df[["label", "clean_document"]].head()

## 2. Train/test split & label encoding

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df["clean_document"], df["label"], test_size=0.3, random_state=RANDOM_STATE, shuffle=True
)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_labels = len(label_encoder.classes_)
target_names = label_encoder.classes_

## 3. Dataset wrapper & classification model

`BertForClassification` wraps any Hugging Face encoder with mean pooling over the last hidden
state + a dropout + linear head, so the same class fine-tunes every checkpoint in `BERT_MODELS`.

In [ ]:
class ClinicalTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

    def __len__(self):
        return len(self.labels)


class BertForClassification(nn.Module):
    def __init__(self, encoder, num_labels, dropout_prob=0.1):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)  # mean pooling over tokens
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

## 4. Fine-tuning + evaluation loop

One function: tokenize with the checkpoint's own tokenizer, fine-tune end to end with AdamW +
cross-entropy for `NUM_EPOCHS` epochs, then evaluate on the held-out test split.

In [ ]:
def fine_tune_and_evaluate(model_name, hf_checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(hf_checkpoint)
    encoder = AutoModel.from_pretrained(hf_checkpoint)

    tokenized_train = tokenizer(list(x_train.values), padding=True, truncation=True,
                                 max_length=MAX_LENGTH, return_tensors="pt")
    tokenized_test = tokenizer(list(x_test.values), padding=True, truncation=True,
                                max_length=MAX_LENGTH, return_tensors="pt")

    train_dataset = ClinicalTextDataset(tokenized_train, torch.tensor(y_train_encoded))
    test_dataset = ClinicalTextDataset(tokenized_test, torch.tensor(y_test_encoded))

    model = BertForClassification(encoder, num_labels).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        with tqdm(total=len(train_dataset), desc=f"{model_name} | epoch {epoch + 1}/{NUM_EPOCHS}", unit="doc") as pbar:
            for i in range(0, len(train_dataset), BATCH_SIZE):
                batch = [train_dataset[j] for j in range(i, min(i + BATCH_SIZE, len(train_dataset)))]
                input_ids = torch.stack([b["input_ids"] for b in batch]).to(DEVICE)
                attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(DEVICE)
                labels = torch.stack([b["labels"] for b in batch]).to(DEVICE)

                optimizer.zero_grad()
                logits = model(input_ids, attention_mask)
                loss = loss_fn(logits, labels)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                pbar.update(len(batch))
                pbar.set_postfix({"loss": loss.item()})
        print(f"{model_name} epoch {epoch + 1}: mean loss {epoch_loss / (len(train_dataset) / BATCH_SIZE):.4f}")

    model.eval()
    predictions = []
    with torch.no_grad():
        for i in range(0, len(test_dataset), BATCH_SIZE):
            batch = [test_dataset[j] for j in range(i, min(i + BATCH_SIZE, len(test_dataset)))]
            input_ids = torch.stack([b["input_ids"] for b in batch]).to(DEVICE)
            attention_mask = torch.stack([b["attention_mask"] for b in batch]).to(DEVICE)
            logits = model(input_ids, attention_mask)
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())

    report = {
        "accuracy": metrics.accuracy_score(y_test_encoded, predictions),
        "precision": metrics.precision_score(y_test_encoded, predictions, average="macro", zero_division=0),
        "recall": metrics.recall_score(y_test_encoded, predictions, average="macro", zero_division=0),
        "f1": metrics.f1_score(y_test_encoded, predictions, average="macro", zero_division=0),
    }
    return model, predictions, report

## 5. Run the sweep & compare (Table 5.14)

In [ ]:
bert_results = {}
bert_models = {}
bert_predictions = {}

for model_name in MODELS_TO_RUN:
    print(f"\n=== Fine-tuning {model_name} ({BERT_MODELS[model_name]}) ===")
    model, predictions, report = fine_tune_and_evaluate(model_name, BERT_MODELS[model_name])
    bert_models[model_name] = model
    bert_predictions[model_name] = predictions
    bert_results[model_name] = report
    print(model_name, report)

bert_results_df = pd.DataFrame(bert_results).T
display(bert_results_df)

In [ ]:
bert_results_df[["accuracy", "f1"]].plot(kind="bar", figsize=(7, 4), title="BERT variant comparison")
plt.ylim(0, 1)
plt.show()

## 6. Best model (ClinicalBERT) — detailed report (Table 5.15)

In [ ]:
best_model_name = "ClinicalBERT" if "ClinicalBERT" in bert_predictions else max(bert_results, key=lambda k: bert_results[k]["f1"])
print("Best model:", best_model_name)

y_pred_best = bert_predictions[best_model_name]
print(metrics.classification_report(y_test_encoded, y_pred_best, target_names=target_names))

cm = metrics.confusion_matrix(y_test_encoded, y_pred_best)
sns.heatmap(cm, center=True, cmap="YlGnBu", xticklabels=target_names, yticklabels=target_names)
plt.title(f"{best_model_name} — confusion matrix")
plt.xticks(rotation=45, ha="right")
plt.show()

## 7. Save the fine-tuned model

In [ ]:
import os

os.makedirs("models", exist_ok=True)
best_model = bert_models[best_model_name]

torch.save(best_model.state_dict(), f"models/{best_model_name.lower()}_finetuned.pt")
print(f"Saved models/{best_model_name.lower()}_finetuned.pt")